In [ ]:

import pyro
from matplotlib import pyplot as plt
import math
import numpy as np
import pandas as pd
import torch

def get_theta_tensor(input_file):
    df = pd.read_csv(input_file)
    num_unique_times = 31
    num_countries = 234
    num_sports = 76

    theta_tensor = torch.zeros((num_unique_times, num_countries, num_sports, 7))

    for row in df.iterrows():
        year = row[1]["Year"]
        country = row[1]["NOC"]
        sport = row[1]["Sport"]
        medal = row[1]["Medal"]
        home = row[1]["Home_Country"]
        experience = row[1]["Experience"]
        experience_std = row[1]["Experience_std"]

        if medal == "Gold":
            theta_tensor[year][country][sport][0] += 1
        elif medal == "Silver":
            theta_tensor[year][country][sport][1] += 1
        elif medal == "Bronze":
            theta_tensor[year][country][sport][2] += 1
        else:
            theta_tensor[year][country][sport][3] += 1
        
        if home == 1:
            theta_tensor[year][country][sport][4] = 1

        theta_tensor[year][country][sport][5] += experience

        theta_tensor[year][country][sport][6] += experience_std

    # Calculate average experience (avoid division by zero)
    num_athlete = torch.clamp(theta_tensor[:,:,:,0:4].sum(dim=3), min=1)
    theta_tensor[:,:,:,5] /= num_athlete
    theta_tensor[:,:,:,6] /= num_athlete

    return theta_tensor


pyro.clear_param_store()
pyro.get_param_store().load("../models/params/trained_model_params_v6.pth")

for name in pyro.get_param_store().keys():
    print("here")
    print(name)

    theta_tensor = get_theta_tensor("../data/processed/summerOly_athletes_team_id_no_high_irregulars_numerical_experience_combined_reassigned_home.csv")

    num_countries = theta_tensor.shape[1]

    for i in range(1,30):

        power_30_table_tennis = []

        # Number of samples for Monte Carlo approximation
        n_samples = 1000

        P_loc_param = pyro.get_param_store()[f"P_loc_{i}"]
        P_loc_param = P_loc_param.detach().numpy()
        P_scale_param = pyro.get_param_store()[f"P_scale_{i}"]
        P_scale_param = P_scale_param.detach().numpy()

        P_loc_table_tennis = P_loc_param[:, 63]
        P_scale_table_tennis = P_scale_param[:, 63]

        for c in range(num_countries): # Iterate over time steps
        # Get parameters for Italy (country index 102) and China (country index 42)
            loc = P_loc_table_tennis[c]
            scale = P_scale_table_tennis[c]
            # Sample from the LogNormal distribution to approximate the mean power
            samples = np.random.lognormal(mean=loc, sigma=scale, size=n_samples)

            # Compute the mean of the samples
            power = np.mean(samples)

            # Append the results to the respective lists
            power_30_table_tennis.append(power)


#Plot scatter plot
plt.scatter(range(num_countries), power_30_table_tennis)
plt.xlabel('Country Index')
plt.ylabel('Power')
plt.title('Power of Countries in Table Tennis at Time Step 30')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'trained_model_params+v6.pth'